In [ ]:
# 식품 영양 정보를 가져와서 csv파일로 저장
# 가져올 정보. 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리
# 내 인증키: 34cf2a75ed8a3fd4404f4da20bb50ee752fd728f079f1dcc0cee0813fe3a649d

In [ ]:
# 필요한 패키지
import requests
import pandas as pd
from bs4 import BeautifulSoup

service_key = '34cf2a75ed8a3fd4404f4da20bb50ee752fd728f079f1dcc0cee0813fe3a649d'

def get_food_nutrition_xml(service_key: str, food_name: str = ''):
  
    url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'
    
    pageNo = 1
    numOfRows = 100
    
    params = {
        'serviceKey': service_key,
        'FOOD_NM_KR': food_name,
        'pageNo': pageNo,
        'numOfRows': numOfRows
    }
    
    # 다양한 예외가 발생하면 각 에러에 맞게 처리를 하기 위해 try를 많이 씀
    try:
        print(f"[{food_name}] 데이터를 수집 중...")
        response = requests.get(url, params=params)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'xml')
        items = soup.find_all('item')
        
        if not items:
            print("검색 결과가 없습니다.")
            return None
            
        food_list = []
        for item in items:
            def get_val(tag_name):
                tag = item.find(tag_name)
                return tag.text if tag and tag.text else "0"

            food_list.append({
                '식품명': get_val('FOOD_NM_KR'),
                '탄수화물(g)': get_val('AMT_NUM2'),
                '단백질(g)': get_val('AMT_NUM3'),
                '지방(g)': get_val('AMT_NUM4'),
                '나트륨(mg)': get_val('AMT_NUM13'),
                '총칼로리(kcal)': get_val('AMT_NUM1'),
                '기준량': get_val('SERVING_SIZE'),
            })
            
        return pd.DataFrame(food_list)

    except Exception as e:
        print(f"오류 발생: {e}")
        return None

def save_to_csv(df, keyword):
    if df is not None:
        file_name = f'food_data_{keyword}.csv'
        df.to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"저장이 완료되었습니다: {file_name} (총 {len(df)}건)")

if __name__ == "__main__":
    MY_KEY = service_key
    keyword = input("검색할 식품명: ")
    
    result_df = get_food_nutrition_xml(MY_KEY, keyword)
    
    if result_df is not None:
        print(result_df)
        save_to_csv(result_df, keyword)

[치즈] 데이터를 수집 중...
                 식품명 탄수화물(g) 단백질(g)  지방(g)  나트륨(mg) 총칼로리(kcal)   기준량
0              김밥_치즈   63.30   6.24   7.03  169.000    177.000  100g
1     볶음밥_갈비천왕 치즈 치밥       0  10.97      0  497.000    208.000  100g
2     볶음밥_볼케이노 치즈 치밥       0  12.81      0  461.000    198.000  100g
3   김치 볶음밥_치즈오븐김치볶음밥       0   8.29   6.36  678.000    160.000  100g
4   김치 볶음밥_치즈오븐김치볶음밥       0   7.87      0  791.000    196.000  100g
..               ...     ...    ...    ...      ...        ...   ...
95          버거_치즈 버거       0  10.81      0  373.000    248.000  100g
96      버거_치즈 버거 (L)       0  10.83      0  388.000    261.000  100g
97    버거_콰트로 맥앤치즈 버거       0  10.76  17.04  574.000    282.000  100g
98   버거_클래식치즈 버거 버터번       0  10.96      0  527.000    337.000  100g
99    버거_트리플딥치즈비프 버거       0  13.55  15.14  411.000    251.000  100g

[100 rows x 7 columns]
저장이 완료되었습니다: food_data_치즈.csv (총 100건)


In [ ]:
# 중복 제거
# result_df = result_df.drop_duplicates() # 행 전체 값이 중복되면 제거
# result_df = result_df.drop_duplicates(subset=['식품명']) # 식품명 열이 중복된 경우만 제거
result_df = result_df.drop_duplicates(subset=['식품명'], keep='first') # 중복되면 첫번째를 살리고 나머지 제거
# result_df = result_df.drop_duplicates(subset=['식품명'], keep='last') # 중복되면 마지막을 살리고 나머지 제거
# result_df = result_df.drop_duplicates(subset=['식품명'], keep='False') # 중복되면 모두 제거

# 결측치 값 변경
result_df['탄수화물(g)'] = result_df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경

# 결측치 제거
result_df = result_df.dropna() # 행에 결측치가 하나라도 있으면 삭제
result_df = result_df(subset=['식품명']) # 식품명에 결측치가 있으면 삭제


In [17]:
service_key = '34cf2a75ed8a3fd4404f4da20bb50ee752fd728f079f1dcc0cee0813fe3a649d'

def get_food_nutrient(service_key:str, page: int=1, food_name: str = '')->pd.DataFrame:
  """공공데이터에서 식품 영양소 db 정보를 가져와서 데이터 프레임으로 반환하는 함수"""
  url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'
    
  page = 1
  numOfRows = 10
    
  params = {
        'serviceKey': service_key,
        'FOOD_NM_KR': food_name,
        'pageNo': page,
        'numOfRows': numOfRows
    }
    
    # 다양한 예외가 발생하면 각 에러에 맞게 처리를 하기 위해 try를 많이 씀
  try:
        print(f"[{food_name}] 데이터를 수집 중...")
        response = requests.get(url, params=params)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'xml')
        items = soup.find_all('item')
        
        if not items:
            print("검색 결과가 없습니다.")
            return None
            
        food_list = []
        for item in items:
            def get_val(tag_name):
                tag = item.find(tag_name)
                return tag.text if tag and tag.text else "0"

            food_list.append({
                '식품명': get_val('FOOD_NM_KR'),
                '탄수화물(g)': get_val('AMT_NUM2'),
                '단백질(g)': get_val('AMT_NUM3'),
                '지방(g)': get_val('AMT_NUM4'),
                '나트륨(mg)': get_val('AMT_NUM13'),
                '총칼로리(kcal)': get_val('AMT_NUM1'),
                '기준량': get_val('SERVING_SIZE'),
            })
            
        return pd.DataFrame(food_list)

  except Exception as e:
        print(f"오류 발생: {e}")
        return None
    
def save_to_csv(df, keyword):
    if df is not None:
        file_name = f'food_data_{keyword}.csv'
        df.to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"저장이 완료되었습니다: {file_name} (총 {len(df)}건)")

def clean_food_nutrient(datas:pd.DataFrame)->pd.DataFrame:
  """식품 영양소 df에서 결측치 제거, 중복 제거하는 함수"""
  
  if datas is None or datas.empty:
        return datas
    
  # 중복 제거
  result_df = datas.drop_duplicates(subset=['식품명'], keep='first') # 중복되면 첫번째를 살리고 나머지 제거

  # 결측치 값 변경
  result_df['탄수화물(g)'] = result_df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경

  # 결측치 제거
  result_df = result_df.dropna() # 행에 결측치가 하나라도 있으면 삭제
  result_df = result_df.dropna(subset=['식품명']) # 식품명에 결측치가 있으면 삭제
  
  return result_df
# food_df = get_food_nutrient(service_key, 1)
# food_df = clean_food_nutrient(food_df)
# print(food_df)

if __name__ == "__main__":
    keyword = input("검색할 식품명: ")
    
    # 수정: service_key를 첫 번째 인자로 명시해주어야 합니다.
    result_df = get_food_nutrient(service_key, food_name=keyword)
    
    # 데이터 전처리 수행
    result_df = clean_food_nutrient(result_df)
    
    if result_df is not None:
        print(result_df)
        save_to_csv(result_df, keyword)

[치즈] 데이터를 수집 중...
                  식품명 탄수화물(g) 단백질(g) 지방(g)  나트륨(mg) 총칼로리(kcal)   기준량
0               김밥_치즈   63.30   6.24  7.03  169.000    177.000  100g
1      볶음밥_갈비천왕 치즈 치밥       0  10.97     0  497.000    208.000  100g
2      볶음밥_볼케이노 치즈 치밥       0  12.81     0  461.000    198.000  100g
3    김치 볶음밥_치즈오븐김치볶음밥       0   8.29  6.36  678.000    160.000  100g
5         계란빵_콘치즈 계란빵       0  10.00     0  225.000    240.000  100g
6         기타빵_슈크림치즈스틱       0  12.14     0  322.000    318.000  100g
7         기타빵_스트링치즈스틱       0  16.93     0  648.000    303.000  100g
8       도넛_뉴욕 치즈케익 도넛       0   5.56     0  167.000    369.000  100g
9  도넛_못난이찹쌀고구마크림치즈 도넛       0   6.81     0  329.000    293.000  100g
저장이 완료되었습니다: food_data_치즈.csv (총 9건)


C:\Users\hi6\AppData\Local\Temp\ipykernel_12468\85678628.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['탄수화물(g)'] = result_df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경
